# Reorganization metrics (cross-patient ranking)

Compute distance matrices for multiple patients and rank metrics by cross-patient consistency.

**Legacy notebooks merged:**
- DSTCMP_all_distance_measures.ipynb
- DSTCMP_permutation_robust.ipynb

In [ ]:
%matplotlib inline
from lrgsglib.config.funcs import move_to_rootf
move_to_rootf(pathname="lrg_eegfc")
from lrg_eegfc.notebook import *

In [ ]:
DATA_ROOT = Path('data/stereoeeg_patients')
patients = list_patients(DATA_ROOT)[:2]
assert patients, 'No patients found under data/stereoeeg_patients'

bands = BRAIN_BANDS_NAMES
phases = list(PHASE_LABELS)
fc_method = 'msc'

from lrg_eegfc.utils.metrics.reorganization import build_metric_specs, compute_metric_matrix
from lrg_eegfc.utils.metrics.comparison import rank_distance_measures

metric_specs = build_metric_specs(distance_metric='euclidean')
measure_results = {key: {band: {} for band in bands} for key in metric_specs}

for patient in patients:
    for band in bands:
        results_by_phase = {
            phase: load_lrg_result(patient, phase, band, fc_method)
            for phase in phases
        }
        if any(r is None for r in results_by_phase.values()):
            continue
        for key, spec in metric_specs.items():
            matrix = compute_metric_matrix(phases, results_by_phase, spec['fn'])
            measure_results[key][band][patient] = matrix

rankings = rank_distance_measures(measure_results, patients=patients, bands=bands)
rankings

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ranked = sorted(
    rankings['scores'].items(),
    key=lambda x: x[1] if np.isfinite(x[1]) else -np.inf,
    reverse=True,
)
labels = [metric_specs[key]['label'] for key, _ in ranked]
scores = [score for _, score in ranked]

fig, ax = plt.subplots(figsize=(10, max(4, 0.4 * len(labels))))
ax.barh(labels, scores, color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Consistency score (avg Pearson)')
ax.set_title('Metric ranking across patients')
plt.tight_layout()
plt.show()